In [24]:
import zarr
import torch
import itertools
import polars as pl
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset

In [2]:
class toy_zarr_dataset(Dataset):
    def __init__(self):
        super().__init__()
        self.store = zarr.storage.ZipStore('/scr/jpeters/parcha/a63fc9b_0.4_mask_prob_features.zip', mode='r')
        self.root = zarr.open(store=self.store)
        self.metatdata = pl.read_csv('/scr/data/CHAMMI/dataset/combined_metadata.csv')['file_path'].to_list()
        
    def load_zarr(self):
        self.store = zarr.storage.ZipStore('/scr/jpeters/parcha/a63fc9b_0.4_mask_prob_features.zip', mode='r')
        self.root = zarr.open(store=self.store)
    
    def __len__(self):
        return len(self.metatdata)
    
    def __getitem__(self, idx):
        return self.root[self.metatdata[idx]][:][:100]

    def worker_init_fn(self):
        worker_info = torch.utils.data.get_worker_info()
        dataset = worker_info.dataset
        dataset.load_zarr()
        
zarr_loader = DataLoader(toy_zarr_dataset(), num_workers=3, batch_size=32, worker_init_fn=toy_zarr_dataset.worker_init_fn)

for idx, bat in tqdm(enumerate(zarr_loader), total=len(zarr_loader)):
    shape = bat.shape

NameError: name 'Dataset' is not defined

In [8]:
store = zarr.storage.ZipStore('/scr/jpeters/parcha/a63fc9b_0.4_mask_prob_features.zip', mode='r')
metatdata = pl.read_csv('/scr/data/CHAMMI/dataset/combined_metadata.csv')

In [15]:
metatdata[0]['file_path'].item()

'HPA/crops/24018_342_G12_1_9.png'

In [3]:
root = zarr.open(store=store)

In [21]:
for filename in tqdm(metatdata['file_path'].to_list()):
    root[filename][:].shape

100%|██████████| 220284/220284 [01:53<00:00, 1948.85it/s]


In [22]:
root.members(max_depth=None)

KeyboardInterrupt: 